# Acidentes no Espaço Aéreo Brasileiro (Cenipa)
Projeto AC2 - Big Data

## Configuração do Dataset

In [1]:
# Importando bibliotecas necessárias
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, isnan, when
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType
from pyspark.ml.feature import VectorAssembler, StringIndexer, OneHotEncoder, StandardScaler
from pyspark.ml.classification import DecisionTreeClassifier, LogisticRegression, MultilayerPerceptronClassifier
from pyspark.ml import Pipeline
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
import unicodedata
import re

spark = SparkSession.builder \
    .appName("Cenipa_BigData_Classification") \
    .getOrCreate()

In [36]:
# Tipando as colunas que vamos utilizar
cenipa_schema = StructType([
    StructField("Codigo da Ocorrencia", IntegerType(), True),
    StructField("Classificacao da Ocorrencia", StringType(), True),
    StructField("Data e Hora da Ocorrencia", StringType(), True),
    StructField("Latitude da Ocorrencia", StringType(), True),
    StructField("Longitude da Ocorrencia", StringType(), True),
    StructField("Cidade da Ocorrencia", StringType(), True),
    StructField("UF da Ocorrencia", StringType(), True),
    StructField("Pais da Ocorrencia", StringType(), True),
    StructField("Aerodromo da Ocorrencia", StringType(), True),
    StructField("Investigacao da Aeronave foi Liberada", StringType(), True),
    StructField("Status da Investigacao", StringType(), True),
    StructField("Numero do Relatorio de Divulgacao", StringType(), True),
    StructField("Relatorio de Divulgacao foi Publicado?", StringType(), True),
    StructField("Dia da Divulgacao da Publicacao", StringType(), True),
    StructField("Total de Recomendacoes", IntegerType(), True),
    StructField("Total de Aeronaves Envolvidas", IntegerType(), True),
    StructField("Ocorrencia na Saida da Pista?", StringType(), True),
    StructField("Tipo da Ocorrencia", StringType(), True),
    StructField("Tipo de Categoria da Ocorrencia", StringType(), True),
    StructField("Taxonomia do Tipo de Icao", StringType(), True),
    StructField("Matricula da Aeronave", StringType(), True),
    StructField("Categoria do Operador de Aeronave", StringType(), True),
    StructField("Tipo de Aeronave", StringType(), True),
    StructField("Fabricante da Aeronave", StringType(), True),
    StructField("Modelo de Aeronave", StringType(), True),
    StructField("Tipo de Aeronave Icao", StringType(), True),
    StructField("Aeronave Motor Tipo", StringType(), True),
    StructField("Aeronave Motor Quantidade", StringType(), True),
    StructField("Aeronave PMD", IntegerType(), True),
    StructField("Categoria PMD Aeronave", IntegerType(), True),
    StructField("Quantidade de Assentos na Aeronave", IntegerType(), True),
    StructField("Ano de Fabricacao da Aeronave", IntegerType(), True),
    StructField("Pais Fabricante da Aeronave", StringType(), True),
    StructField("Pais de Registro da Aeronave", StringType(), True),
    StructField("Registro da Categoria da Aeronave", StringType(), True),
    StructField("Registro Segmento da Aeronave", StringType(), True),
    StructField("Voo de Origem do Acidente", StringType(), True),
    StructField("Voo Destino do Acidente", StringType(), True),
    StructField("Fase de Operacao da Aeronave", StringType(), True),
    StructField("Tipo de Operacao da Aeronave", StringType(), True),
    StructField("Nivel de Dano da Aeronave", StringType(), True),
    StructField("Total de Fatalidades no Acidente", IntegerType(), True),
    StructField("Fator do Acidente", StringType(), True),
    StructField("Aspecto do Fator da Ocorrencia", StringType(), True),
    StructField("Fator Condicionante", StringType(), True),
    StructField("Area do Fator", StringType(), True),
    StructField("Numero da Recomendacao", StringType(), True),
    StructField("Dia da Assinatura de Recomendacao", StringType(), True),
    StructField("Dia do Encaminhamento da Recomendacao", StringType(), True),
    StructField("Conteudo da Recomendacao", StringType(), True),
    StructField("Status da Recomendacao", StringType(), True),
    StructField("Orgao de Recomendacao", StringType(), True),
])

dataframe = spark.read.csv("./assets/data/Cenipa.csv", header=True, schema=cenipa_schema, sep=";")

num_linhas = dataframe.count()
print(f"Número de linhas no DataFrame: {num_linhas}")

Número de linhas no DataFrame: 6114


## Análise de Dados

In [3]:
# Visualizando o esquema dos dados
dataframe.printSchema()

root
 |-- Codigo da Ocorrencia: integer (nullable = true)
 |-- Classificacao da Ocorrencia: string (nullable = true)
 |-- Data e Hora da Ocorrencia: string (nullable = true)
 |-- Latitude da Ocorrencia: string (nullable = true)
 |-- Longitude da Ocorrencia: string (nullable = true)
 |-- Cidade da Ocorrencia: string (nullable = true)
 |-- UF da Ocorrencia: string (nullable = true)
 |-- Pais da Ocorrencia: string (nullable = true)
 |-- Aerodromo da Ocorrencia: string (nullable = true)
 |-- Investigacao da Aeronave foi Liberada: string (nullable = true)
 |-- Status da Investigacao: string (nullable = true)
 |-- Numero do Relatorio de Divulgacao: string (nullable = true)
 |-- Relatorio de Divulgacao foi Publicado?: string (nullable = true)
 |-- Dia da Divulgacao da Publicacao: string (nullable = true)
 |-- Total de Recomendacoes: integer (nullable = true)
 |-- Total de Aeronaves Envolvidas: integer (nullable = true)
 |-- Ocorrencia na Saida da Pista?: string (nullable = true)
 |-- Tipo da 

In [4]:
dataframe.show(5)

+--------------------+---------------------------+-------------------------+----------------------+-----------------------+--------------------+----------------+------------------+-----------------------+-------------------------------------+----------------------+---------------------------------+--------------------------------------+-------------------------------+----------------------+-----------------------------+-----------------------------+--------------------+-------------------------------+-------------------------+---------------------+---------------------------------+----------------+----------------------+------------------+---------------------+-------------------+-------------------------+------------+----------------------+----------------------------------+-----------------------------+---------------------------+----------------------------+---------------------------------+-----------------------------+-------------------------+-----------------------+-------------

In [5]:
colunas_numericas = [col_name for col_name, data_type in dataframe.dtypes if data_type in ('int', 'double')]
print("Apenas colunas numéricas:", colunas_numericas)

Apenas colunas numéricas: ['Codigo da Ocorrencia', 'Total de Recomendacoes', 'Total de Aeronaves Envolvidas', 'Aeronave PMD', 'Categoria PMD Aeronave', 'Quantidade de Assentos na Aeronave', 'Ano de Fabricacao da Aeronave', 'Total de Fatalidades no Acidente']


In [6]:
# Verificando a distribuição da classe alvo "Classificacao da Ocorrencia"
print("Distribuição da classe alvo (Classificacao da Ocorrencia):")
distribuicao_classes = dataframe.groupBy("Classificacao da Ocorrencia").count().orderBy(col("count").desc())
distribuicao_classes.show()

Distribuição da classe alvo (Classificacao da Ocorrencia):
+---------------------------+-----+
|Classificacao da Ocorrencia|count|
+---------------------------+-----+
|                  INCIDENTE| 3393|
|                   ACIDENTE| 1930|
|            INCIDENTE GRAVE|  791|
+---------------------------+-----+



In [7]:
# Registrando o DataFrame como uma tabela SQL chamada "cenipa_data"
dataframe.createOrReplaceTempView("cenipa_data")

In [8]:
print("Analisando cardinalidade para encontrar colunas identificadoras (IDs):")
spark.sql("""
    SELECT 
        COUNT(*) AS total_linhas,
        COUNT(DISTINCT `Codigo da Ocorrencia`) AS dist_codigo,
        COUNT(DISTINCT `Matricula da Aeronave`) AS dist_matricula,
        COUNT(DISTINCT `Data e Hora da Ocorrencia`) AS dist_data_hora
    FROM cenipa_data
""").show()

Analisando cardinalidade para encontrar colunas identificadoras (IDs):
+------------+-----------+--------------+--------------+
|total_linhas|dist_codigo|dist_matricula|dist_data_hora|
+------------+-----------+--------------+--------------+
|        6114|       6114|          4419|          6090|
+------------+-----------+--------------+--------------+



In [9]:
print("Distribuição da classe (Matricula da Aeronave):")
distribuicao_classes = dataframe.groupBy("Matricula da Aeronave").count().orderBy(col("count").desc())
distribuicao_classes.show()

Distribuição da classe (Matricula da Aeronave):
+---------------------+-----+
|Matricula da Aeronave|count|
+---------------------+-----+
|                *****|   14|
|                PPGMA|   10|
|                PRTTK|   10|
|                PRTTP|   10|
|                PRTTW|    9|
|                PPFXH|    9|
|                PRFLM|    9|
|                PPPTO|    9|
|                PPGOB|    8|
|                PTLSJ|    8|
|                PPGBC|    8|
|                PRAYN|    8|
|                PTBKU|    8|
|                PRAZA|    8|
|                PRATV|    8|
|                PRAZC|    8|
|                PPPTQ|    7|
|                PREJI|    7|
|                PRTTO|    7|
|                PROAM|    7|
+---------------------+-----+
only showing top 20 rows



In [10]:
print("Relação entre Nível de Dano e Classificação da Ocorrência:")
spark.sql("""
    SELECT 
        `Nivel de Dano da Aeronave`,
        `Classificacao da Ocorrencia`,
        COUNT(*) AS total
    FROM cenipa_data
    GROUP BY `Nivel de Dano da Aeronave`, `Classificacao da Ocorrencia`
    ORDER BY `Nivel de Dano da Aeronave`, total DESC
""").show(30)

Relação entre Nível de Dano e Classificação da Ocorrência:
+-------------------------+---------------------------+-----+
|Nivel de Dano da Aeronave|Classificacao da Ocorrencia|total|
+-------------------------+---------------------------+-----+
|                      ***|                   ACIDENTE|   34|
|                      ***|                  INCIDENTE|   13|
|                      ***|            INCIDENTE GRAVE|    6|
|                DESTRUÍDA|                   ACIDENTE|  365|
|                     LEVE|                  INCIDENTE| 1362|
|                     LEVE|            INCIDENTE GRAVE|  409|
|                     LEVE|                   ACIDENTE|   52|
|                   NENHUM|                  INCIDENTE| 1955|
|                   NENHUM|            INCIDENTE GRAVE|  185|
|                   NENHUM|                   ACIDENTE|   27|
|              SUBSTANCIAL|                   ACIDENTE| 1452|
|              SUBSTANCIAL|            INCIDENTE GRAVE|  191|
|          

In [11]:
print("Relação entre Fase de Operacao da Aeronave e Classificação da Ocorrência:")
spark.sql("""
    SELECT 
        `Fase de Operacao da Aeronave`,
        `Classificacao da Ocorrencia`,
        COUNT(*) AS total
    FROM cenipa_data
    GROUP BY `Fase de Operacao da Aeronave`, `Classificacao da Ocorrencia`
    ORDER BY `Fase de Operacao da Aeronave`, total DESC
""").show(30)

Relação entre Fase de Operacao da Aeronave e Classificação da Ocorrência:
+----------------------------+---------------------------+-----+
|Fase de Operacao da Aeronave|Classificacao da Ocorrencia|total|
+----------------------------+---------------------------+-----+
|                         ***|                   ACIDENTE|   19|
|                         ***|                  INCIDENTE|    6|
|                         ***|            INCIDENTE GRAVE|    1|
|           APROXIMAÇÃO FINAL|                  INCIDENTE|  211|
|           APROXIMAÇÃO FINAL|                   ACIDENTE|   75|
|           APROXIMAÇÃO FINAL|            INCIDENTE GRAVE|   26|
|            ARREMETIDA NO AR|                   ACIDENTE|   19|
|            ARREMETIDA NO AR|                  INCIDENTE|    9|
|            ARREMETIDA NO AR|            INCIDENTE GRAVE|    3|
|          ARREMETIDA NO SOLO|                   ACIDENTE|   20|
|          ARREMETIDA NO SOLO|                  INCIDENTE|   16|
|          ARREM

In [12]:
print("Relação entre Tipo da Ocorrencia e Classificação da Ocorrência:")
spark.sql("""
    SELECT 
        `Tipo da Ocorrencia`,
        `Classificacao da Ocorrencia`,
        COUNT(*) AS total
    FROM cenipa_data
    GROUP BY `Tipo da Ocorrencia`, `Classificacao da Ocorrencia`
    ORDER BY `Tipo da Ocorrencia`, total DESC
""").show(30)

Relação entre Tipo da Ocorrencia e Classificação da Ocorrência:
+--------------------+---------------------------+-----+
|  Tipo da Ocorrencia|Classificacao da Ocorrencia|total|
+--------------------+---------------------------+-----+
|AERONAVE ATINGIDA...|                  INCIDENTE|   11|
|AERONAVE ATINGIDA...|                   ACIDENTE|    1|
|           AERÓDROMO|                  INCIDENTE|    3|
|           AERÓDROMO|            INCIDENTE GRAVE|    2|
|           AERÓDROMO|                   ACIDENTE|    1|
|ALARME FALSO DE F...|                  INCIDENTE|   13|
|ALARME FALSO DE F...|                   ACIDENTE|    1|
|CAUSADO POR FENÔM...|                  INCIDENTE|   83|
|CAUSADO POR FENÔM...|                   ACIDENTE|   19|
|CAUSADO POR FENÔM...|            INCIDENTE GRAVE|    1|
|CAUSADO POR FENÔM...|                   ACIDENTE|    7|
|CAUSADO POR FENÔM...|                  INCIDENTE|    4|
|CAUSADO POR FENÔM...|            INCIDENTE GRAVE|    3|
|CAUSADO POR RICOC...|  

In [13]:
print("Relação entre UF da Ocorrencia e Classificação da Ocorrência:")
spark.sql("""
    SELECT 
        `UF da Ocorrencia`,
        `Classificacao da Ocorrencia`,
        COUNT(*) AS total
    FROM cenipa_data
    GROUP BY `UF da Ocorrencia`, `Classificacao da Ocorrencia`
    ORDER BY `UF da Ocorrencia`, total DESC
""").show(30)

Relação entre UF da Ocorrencia e Classificação da Ocorrência:
+----------------+---------------------------+-----+
|UF da Ocorrencia|Classificacao da Ocorrencia|total|
+----------------+---------------------------+-----+
|             ***|                   ACIDENTE|    2|
|              AC|                  INCIDENTE|   38|
|              AC|                   ACIDENTE|   14|
|              AC|            INCIDENTE GRAVE|   10|
|              AL|                  INCIDENTE|   24|
|              AL|            INCIDENTE GRAVE|    6|
|              AL|                   ACIDENTE|    4|
|              AM|                  INCIDENTE|  148|
|              AM|                   ACIDENTE|   70|
|              AM|            INCIDENTE GRAVE|   27|
|              AP|                  INCIDENTE|    8|
|              AP|                   ACIDENTE|    5|
|              AP|            INCIDENTE GRAVE|    1|
|              BA|                  INCIDENTE|  127|
|              BA|                   

In [14]:
print("Distribuição do Pais da Ocorrencia:")
distribuicao_classes = dataframe.groupBy("Pais da Ocorrencia").count().orderBy(col("count").desc())
distribuicao_classes.show()

print("Distribuição do UF da Ocorrencia:")
distribuicao_classes = dataframe.groupBy("UF da Ocorrencia").count().orderBy(col("count").desc())
distribuicao_classes.show()

print("Distribuição do Aerodromo da Ocorrencia:")
distribuicao_classes = dataframe.groupBy("Aerodromo da Ocorrencia").count().orderBy(col("count").desc())
distribuicao_classes.show()

print("Distribuição do Total de Recomendacoes:")
distribuicao_classes = dataframe.groupBy("Total de Recomendacoes").count().orderBy(col("count").desc())
distribuicao_classes.show()

print("Distribuição do Total de Aeronaves Envolvidas:")
distribuicao_classes = dataframe.groupBy("Total de Aeronaves Envolvidas").count().orderBy(col("count").desc())
distribuicao_classes.show()

Distribuição do Pais da Ocorrencia:
+------------------+-----+
|Pais da Ocorrencia|count|
+------------------+-----+
|            BRASIL| 6114|
+------------------+-----+

Distribuição do UF da Ocorrencia:
+----------------+-----+
|UF da Ocorrencia|count|
+----------------+-----+
|              SP| 1464|
|              MG|  570|
|              RJ|  536|
|              PR|  502|
|              RS|  371|
|              GO|  333|
|              MT|  320|
|              PA|  305|
|              AM|  245|
|              BA|  230|
|              SC|  196|
|              MS|  168|
|              DF|  142|
|              PE|  107|
|              CE|   90|
|              ES|   83|
|              MA|   76|
|              RR|   62|
|              AC|   62|
|              TO|   55|
+----------------+-----+
only showing top 20 rows

Distribuição do Aerodromo da Ocorrencia:
+-----------------------+-----+
|Aerodromo da Ocorrencia|count|
+-----------------------+-----+
|                   ****| 2272|

In [15]:
print("Relação entre Total de Recomendacoes e Classificação da Ocorrência:")
spark.sql("""
    SELECT 
        `Total de Recomendacoes`,
        `Classificacao da Ocorrencia`,
        COUNT(*) AS total
    FROM cenipa_data
    GROUP BY `Total de Recomendacoes`, `Classificacao da Ocorrencia`
    ORDER BY `Total de Recomendacoes`, total DESC
""").show(30)

Relação entre Total de Recomendacoes e Classificação da Ocorrência:
+----------------------+---------------------------+-----+
|Total de Recomendacoes|Classificacao da Ocorrencia|total|
+----------------------+---------------------------+-----+
|                     0|                  INCIDENTE| 3370|
|                     0|                   ACIDENTE| 1348|
|                     0|            INCIDENTE GRAVE|  623|
|                     1|                   ACIDENTE|  254|
|                     1|            INCIDENTE GRAVE|   73|
|                     1|                  INCIDENTE|    7|
|                     2|                   ACIDENTE|  165|
|                     2|            INCIDENTE GRAVE|   43|
|                     2|                  INCIDENTE|    3|
|                     3|                   ACIDENTE|   65|
|                     3|            INCIDENTE GRAVE|   26|
|                     3|                  INCIDENTE|    4|
|                     4|                   ACID

In [16]:
print("Relação entre Total de Aeronaves Envolvidas e Classificação da Ocorrência:")
spark.sql("""
    SELECT 
        `Total de Aeronaves Envolvidas`,
        `Classificacao da Ocorrencia`,
        COUNT(*) AS total
    FROM cenipa_data
    GROUP BY `Total de Aeronaves Envolvidas`, `Classificacao da Ocorrencia`
    ORDER BY `Total de Aeronaves Envolvidas`, total DESC
""").show(30)

Relação entre Total de Aeronaves Envolvidas e Classificação da Ocorrência:
+-----------------------------+---------------------------+-----+
|Total de Aeronaves Envolvidas|Classificacao da Ocorrencia|total|
+-----------------------------+---------------------------+-----+
|                            1|                  INCIDENTE| 3356|
|                            1|                   ACIDENTE| 1917|
|                            1|            INCIDENTE GRAVE|  770|
|                            2|                  INCIDENTE|   35|
|                            2|            INCIDENTE GRAVE|   20|
|                            2|                   ACIDENTE|   13|
|                            3|                  INCIDENTE|    2|
|                            3|            INCIDENTE GRAVE|    1|
+-----------------------------+---------------------------+-----+



In [17]:
print("Relação entre Ocorrencia na Saida da Pista? e Classificação da Ocorrência:")
spark.sql("""
    SELECT 
        `Ocorrencia na Saida da Pista?`,
        `Classificacao da Ocorrencia`,
        COUNT(*) AS total
    FROM cenipa_data
    GROUP BY `Ocorrencia na Saida da Pista?`, `Classificacao da Ocorrencia`
    ORDER BY `Ocorrencia na Saida da Pista?`, total DESC
""").show(30)

Relação entre Ocorrencia na Saida da Pista? e Classificação da Ocorrência:
+-----------------------------+---------------------------+-----+
|Ocorrencia na Saida da Pista?|Classificacao da Ocorrencia|total|
+-----------------------------+---------------------------+-----+
|                          NÃO|                  INCIDENTE| 3334|
|                          NÃO|                   ACIDENTE| 1655|
|                          NÃO|            INCIDENTE GRAVE|  587|
|                          SIM|                   ACIDENTE|  275|
|                          SIM|            INCIDENTE GRAVE|  204|
|                          SIM|                  INCIDENTE|   59|
+-----------------------------+---------------------------+-----+



In [18]:
print("Relação entre Quantidade de Assentos na Aeronave e Classificação da Ocorrência:")
spark.sql("""
    SELECT 
        `Quantidade de Assentos na Aeronave`,
        `Classificacao da Ocorrencia`,
        COUNT(*) AS total
    FROM cenipa_data
    GROUP BY `Quantidade de Assentos na Aeronave`, `Classificacao da Ocorrencia`
    ORDER BY `Quantidade de Assentos na Aeronave`, total DESC
""").show(30)

Relação entre Quantidade de Assentos na Aeronave e Classificação da Ocorrência:
+----------------------------------+---------------------------+-----+
|Quantidade de Assentos na Aeronave|Classificacao da Ocorrencia|total|
+----------------------------------+---------------------------+-----+
|                              NULL|                  INCIDENTE|  103|
|                              NULL|                   ACIDENTE|   62|
|                              NULL|            INCIDENTE GRAVE|   12|
|                                 0|                  INCIDENTE|  139|
|                                 0|                   ACIDENTE|   94|
|                                 0|            INCIDENTE GRAVE|   28|
|                                 1|                   ACIDENTE|  456|
|                                 1|            INCIDENTE GRAVE|   82|
|                                 1|                  INCIDENTE|   64|
|                                 2|                   ACIDENTE|  42

In [19]:
print("Relação entre Ano de Fabricacao da Aeronave e Classificação da Ocorrência:")
spark.sql("""
    SELECT 
        `Ano de Fabricacao da Aeronave`,
        `Classificacao da Ocorrencia`,
        COUNT(*) AS total
    FROM cenipa_data
    GROUP BY `Ano de Fabricacao da Aeronave`, `Classificacao da Ocorrencia`
    ORDER BY `Ano de Fabricacao da Aeronave`, total DESC
""").show(30)

Relação entre Ano de Fabricacao da Aeronave e Classificação da Ocorrência:
+-----------------------------+---------------------------+-----+
|Ano de Fabricacao da Aeronave|Classificacao da Ocorrencia|total|
+-----------------------------+---------------------------+-----+
|                         NULL|                  INCIDENTE|  124|
|                         NULL|                   ACIDENTE|   45|
|                         NULL|            INCIDENTE GRAVE|   13|
|                            0|                  INCIDENTE|  159|
|                            0|                   ACIDENTE|  109|
|                            0|            INCIDENTE GRAVE|   36|
|                         1936|                   ACIDENTE|    1|
|                         1936|                  INCIDENTE|    1|
|                         1940|            INCIDENTE GRAVE|    1|
|                         1942|                  INCIDENTE|    1|
|                         1945|                   ACIDENTE|    2|
|

In [20]:
print("Relação entre Total de Fatalidades no Acidente e Classificação da Ocorrência:")
spark.sql("""
    SELECT 
        `Total de Fatalidades no Acidente`,
        `Classificacao da Ocorrencia`,
        COUNT(*) AS total
    FROM cenipa_data
    GROUP BY `Total de Fatalidades no Acidente`, `Classificacao da Ocorrencia`
    ORDER BY `Total de Fatalidades no Acidente`, total DESC
""").show(30)

Relação entre Total de Fatalidades no Acidente e Classificação da Ocorrência:
+--------------------------------+---------------------------+-----+
|Total de Fatalidades no Acidente|Classificacao da Ocorrencia|total|
+--------------------------------+---------------------------+-----+
|                               0|                  INCIDENTE| 3393|
|                               0|                   ACIDENTE| 1477|
|                               0|            INCIDENTE GRAVE|  791|
|                               1|                   ACIDENTE|  230|
|                               2|                   ACIDENTE|  126|
|                               3|                   ACIDENTE|   33|
|                               4|                   ACIDENTE|   31|
|                               5|                   ACIDENTE|   17|
|                               6|                   ACIDENTE|    7|
|                               7|                   ACIDENTE|    4|
|                        

In [21]:
print("Distribuição do Pais Fabricante da Aeronave:")
distribuicao_classes = dataframe.groupBy("Pais Fabricante da Aeronave").count().orderBy(col("count").desc())
distribuicao_classes.show()

print("Distribuição do Pais de Registro da Aeronave:")
distribuicao_classes = dataframe.groupBy("Pais de Registro da Aeronave").count().orderBy(col("count").desc())
distribuicao_classes.show()

print("Distribuição da Area do Fator:")
distribuicao_classes = dataframe.groupBy("Area do Fator").count().orderBy(col("count").desc())
distribuicao_classes.show()

Distribuição do Pais Fabricante da Aeronave:
+---------------------------+-----+
|Pais Fabricante da Aeronave|count|
+---------------------------+-----+
|                     BRASIL| 6003|
|             ESTADOS UNIDOS|   51|
|           NÃO IDENTIFICADO|   12|
|                   PARAGUAI|    8|
|                      CHILE|    5|
|                   PORTUGAL|    5|
|                  ARGENTINA|    4|
|                    BOLÍVIA|    4|
|                   ALEMANHA|    3|
|                    ESPANHA|    3|
|                     PANAMÁ|    2|
|                   COLÔMBIA|    2|
|                     FRANÇA|    2|
|                      SUIÇA|    1|
|            EMIRADOS ÁRABES|    1|
|                    URUGUAI|    1|
|                     ITÁLIA|    1|
|                  CINGAPURA|    1|
|                    HOLANDA|    1|
|                     RÚSSIA|    1|
+---------------------------+-----+
only showing top 20 rows

Distribuição do Pais de Registro da Aeronave:
+------------------

In [22]:
print("Relação entre Tipo de Aeronave e Classificação da Ocorrência:")
spark.sql("""
    SELECT 
        `Tipo de Aeronave`,
        `Classificacao da Ocorrencia`,
        COUNT(*) AS total
    FROM cenipa_data
    GROUP BY `Tipo de Aeronave`, `Classificacao da Ocorrencia`
    ORDER BY `Tipo de Aeronave`, total DESC
""").show(30)

Relação entre Tipo de Aeronave e Classificação da Ocorrência:
+----------------+---------------------------+-----+
|Tipo de Aeronave|Classificacao da Ocorrencia|total|
+----------------+---------------------------+-----+
|             ***|                  INCIDENTE|   82|
|             ***|                   ACIDENTE|   44|
|             ***|            INCIDENTE GRAVE|   14|
|         ANFÍBIO|                   ACIDENTE|    7|
|         ANFÍBIO|                  INCIDENTE|    7|
|         ANFÍBIO|            INCIDENTE GRAVE|    1|
|           AVIÃO|                  INCIDENTE| 2860|
|           AVIÃO|                   ACIDENTE| 1385|
|           AVIÃO|            INCIDENTE GRAVE|  668|
|           BALÃO|                   ACIDENTE|    1|
|       DIRIGÍVEL|                  INCIDENTE|    1|
|     HELICÓPTERO|                  INCIDENTE|  381|
|     HELICÓPTERO|                   ACIDENTE|  236|
|     HELICÓPTERO|            INCIDENTE GRAVE|   63|
|      HIDROAVIÃO|                  I

In [23]:
print("Relação entre Tipo de Operacao da Aeronave e Classificação da Ocorrência:")
spark.sql("""
    SELECT 
        `Tipo de Operacao da Aeronave`,
        `Classificacao da Ocorrencia`,
        COUNT(*) AS total
    FROM cenipa_data
    GROUP BY `Tipo de Operacao da Aeronave`, `Classificacao da Ocorrencia`
    ORDER BY `Tipo de Operacao da Aeronave`, total DESC
""").show(30)

Relação entre Tipo de Operacao da Aeronave e Classificação da Ocorrência:
+----------------------------+---------------------------+-----+
|Tipo de Operacao da Aeronave|Classificacao da Ocorrencia|total|
+----------------------------+---------------------------+-----+
|                         ***|                  INCIDENTE|  103|
|                         ***|            INCIDENTE GRAVE|   29|
|                         ***|                   ACIDENTE|   23|
|                    AGRÍCOLA|                   ACIDENTE|  410|
|                    AGRÍCOLA|            INCIDENTE GRAVE|   64|
|                    AGRÍCOLA|                  INCIDENTE|   35|
|               ESPECIALIZADA|                  INCIDENTE|   44|
|               ESPECIALIZADA|                   ACIDENTE|   26|
|               ESPECIALIZADA|            INCIDENTE GRAVE|   13|
|                EXPERIMENTAL|                   ACIDENTE|  190|
|                EXPERIMENTAL|                  INCIDENTE|   66|
|               

In [38]:
print("Relação entre Categoria PMD Aeronave e Classificação da Ocorrência:")
spark.sql("""
    SELECT 
        `Aeronave PMD`,
        `Classificacao da Ocorrencia`,
        COUNT(*) AS total
    FROM cenipa_data
    GROUP BY `Aeronave PMD`, `Classificacao da Ocorrencia`
    ORDER BY `Aeronave PMD`, total DESC
""").show(30)

Relação entre Categoria PMD Aeronave e Classificação da Ocorrência:
+------------+---------------------------+-----+
|Aeronave PMD|Classificacao da Ocorrencia|total|
+------------+---------------------------+-----+
|           0|                  INCIDENTE|  115|
|           0|                   ACIDENTE|   77|
|           0|            INCIDENTE GRAVE|   24|
|          35|                   ACIDENTE|    1|
|         208|                   ACIDENTE|    1|
|         250|                  INCIDENTE|    1|
|         250|                   ACIDENTE|    1|
|         280|                   ACIDENTE|    3|
|         300|                   ACIDENTE|    1|
|         308|                   ACIDENTE|    2|
|         340|                   ACIDENTE|    1|
|         342|                  INCIDENTE|    1|
|         352|                   ACIDENTE|    1|
|         355|                   ACIDENTE|    2|
|         360|                   ACIDENTE|    2|
|         365|                   ACIDENTE|    1|
|

## Limpando o Dataset

In [39]:
colunas_para_remover = [
    "Codigo da Ocorrencia", # Identificador único
    "Matricula da Aeronave",
    "Data e Hora da Ocorrencia",
    "Latitude da Ocorrencia",   # Coordenada crua
    "Longitude da Ocorrencia",   # Coordenada crua
    "Conteudo da Recomendacao",  # Ocorre DEPOIS do acidente
    "Status da Recomendacao",    # Ocorre DEPOIS do acidente
    "Orgao de Recomendacao",
    "Investigacao da Aeronave foi Liberada",
    "Status da Investigacao",
    "Numero do Relatorio de Divulgacao",
    "Relatorio de Divulgacao foi Publicado?",
    "Dia da Divulgacao da Publicacao",
    "Numero da Recomendacao",
    "Dia da Assinatura de Recomendacao",
    "Dia do Encaminhamento da Recomendacao",
    "Pais da Ocorrencia", # Valor único
    "Cidade da Ocorrencia",
    "Aerodromo da Ocorrencia",
    "Ano de Fabricacao da Aeronave",
    "Quantidade de Assentos na Aeronave",
    "Pais Fabricante da Aeronave",
    "Pais de Registro da Aeronave"
]

df_pre_processado = dataframe.drop(*colunas_para_remover)

print(f"Número de colunas ANTES do drop: {len(dataframe.columns)}")
print(f"Número de colunas DEPOIS do drop: {len(df_pre_processado.columns)}")

# Mostrando o novo schema para confirmar
df_pre_processado.printSchema()

Número de colunas ANTES do drop: 52
Número de colunas DEPOIS do drop: 29
root
 |-- Classificacao da Ocorrencia: string (nullable = true)
 |-- UF da Ocorrencia: string (nullable = true)
 |-- Total de Recomendacoes: integer (nullable = true)
 |-- Total de Aeronaves Envolvidas: integer (nullable = true)
 |-- Ocorrencia na Saida da Pista?: string (nullable = true)
 |-- Tipo da Ocorrencia: string (nullable = true)
 |-- Tipo de Categoria da Ocorrencia: string (nullable = true)
 |-- Taxonomia do Tipo de Icao: string (nullable = true)
 |-- Categoria do Operador de Aeronave: string (nullable = true)
 |-- Tipo de Aeronave: string (nullable = true)
 |-- Fabricante da Aeronave: string (nullable = true)
 |-- Modelo de Aeronave: string (nullable = true)
 |-- Tipo de Aeronave Icao: string (nullable = true)
 |-- Aeronave Motor Tipo: string (nullable = true)
 |-- Aeronave Motor Quantidade: string (nullable = true)
 |-- Aeronave PMD: integer (nullable = true)
 |-- Categoria PMD Aeronave: integer (nullab

In [40]:
df_pre_processado.show(5)

+---------------------------+----------------+----------------------+-----------------------------+-----------------------------+--------------------+-------------------------------+-------------------------+---------------------------------+----------------+----------------------+------------------+---------------------+-------------------+-------------------------+------------+----------------------+---------------------------------+-----------------------------+-------------------------+-----------------------+----------------------------+----------------------------+-------------------------+--------------------------------+--------------------+------------------------------+--------------------+-----------------+
|Classificacao da Ocorrencia|UF da Ocorrencia|Total de Recomendacoes|Total de Aeronaves Envolvidas|Ocorrencia na Saida da Pista?|  Tipo da Ocorrencia|Tipo de Categoria da Ocorrencia|Taxonomia do Tipo de Icao|Categoria do Operador de Aeronave|Tipo de Aeronave|Fabricante da

In [41]:
# Função para limpar e normalizar strings
def normalizar_nome_coluna(nome):
    # Remove acentos
    nome_sem_acento = unicodedata.normalize('NFKD', nome).encode('ASCII', 'ignore').decode('utf-8')
    # Substitui espaços e hifens por underline (_) e deixa tudo em minúsculo
    nome_formatado = re.sub(r'[\s\-]+', '_', nome_sem_acento.strip()).lower()
    # Remove quaisquer outros caracteres especiais que tenham sobrado
    nome_limpo = re.sub(r'[^\w]', '', nome_formatado)
    return nome_limpo

# Cria uma lista com os novos nomes das colunas
novas_colunas = [normalizar_nome_coluna(c) for c in df_pre_processado.columns]

# Aplica os novos nomes ao DataFrame
dataframe = df_pre_processado.toDF(*novas_colunas)

print("Novas colunas normalizadas:")
print(dataframe.columns)

Novas colunas normalizadas:
['classificacao_da_ocorrencia', 'uf_da_ocorrencia', 'total_de_recomendacoes', 'total_de_aeronaves_envolvidas', 'ocorrencia_na_saida_da_pista', 'tipo_da_ocorrencia', 'tipo_de_categoria_da_ocorrencia', 'taxonomia_do_tipo_de_icao', 'categoria_do_operador_de_aeronave', 'tipo_de_aeronave', 'fabricante_da_aeronave', 'modelo_de_aeronave', 'tipo_de_aeronave_icao', 'aeronave_motor_tipo', 'aeronave_motor_quantidade', 'aeronave_pmd', 'categoria_pmd_aeronave', 'registro_da_categoria_da_aeronave', 'registro_segmento_da_aeronave', 'voo_de_origem_do_acidente', 'voo_destino_do_acidente', 'fase_de_operacao_da_aeronave', 'tipo_de_operacao_da_aeronave', 'nivel_de_dano_da_aeronave', 'total_de_fatalidades_no_acidente', 'fator_do_acidente', 'aspecto_do_fator_da_ocorrencia', 'fator_condicionante', 'area_do_fator']


In [42]:
# Tratando valores nulos
# dataframe = dataframe.dropna()

num_linhas = dataframe.count()
print(f"Número de linhas no DataFrame: {num_linhas}")

Número de linhas no DataFrame: 6114


In [43]:
dataframe.printSchema()

root
 |-- classificacao_da_ocorrencia: string (nullable = true)
 |-- uf_da_ocorrencia: string (nullable = true)
 |-- total_de_recomendacoes: integer (nullable = true)
 |-- total_de_aeronaves_envolvidas: integer (nullable = true)
 |-- ocorrencia_na_saida_da_pista: string (nullable = true)
 |-- tipo_da_ocorrencia: string (nullable = true)
 |-- tipo_de_categoria_da_ocorrencia: string (nullable = true)
 |-- taxonomia_do_tipo_de_icao: string (nullable = true)
 |-- categoria_do_operador_de_aeronave: string (nullable = true)
 |-- tipo_de_aeronave: string (nullable = true)
 |-- fabricante_da_aeronave: string (nullable = true)
 |-- modelo_de_aeronave: string (nullable = true)
 |-- tipo_de_aeronave_icao: string (nullable = true)
 |-- aeronave_motor_tipo: string (nullable = true)
 |-- aeronave_motor_quantidade: string (nullable = true)
 |-- aeronave_pmd: integer (nullable = true)
 |-- categoria_pmd_aeronave: integer (nullable = true)
 |-- registro_da_categoria_da_aeronave: string (nullable = true

## Pré-Processamento e Tratamento de Dados

In [44]:
# 1. Removendo registros nulos na coluna alvo
df_clean = dataframe.dropna(subset=["classificacao_da_ocorrencia"])

# 2. Convertendo a coluna alvo (String) para valores numéricos (Double)
label_indexer = StringIndexer(inputCol="classificacao_da_ocorrencia", outputCol="label")
df_clean = label_indexer.fit(df_clean).transform(df_clean)

# 3. Tratamento de classes desbalanceadas (Undersampling)
# Calculamos a proporção para igualar todas as classes pela classe minoritária
class_counts = df_clean.groupBy('label').count().collect()
min_count = min([row['count'] for row in class_counts])
fractions = {row['label']: min_count / row['count'] for row in class_counts}

df_balanced = df_clean.sampleBy('label', fractions, seed=42)
print("Distribuição após o balanceamento das classes:")
df_balanced.groupBy("label").count().show()

# 4. Seleção e conversão de features
colunas_ignorar = ["classificacao_da_ocorrencia", "label"]

# categorical_cols = ["tipo_da_ocorrencia", "fase_de_operacao_da_aeronave", "nivel_de_dano_da_aeronave", "total_de_aeronaves_envolvidas", "ocorrencia_na_saida_da_pista", "total_de_fatalidades_no_acidente", "tipo_de_aeronave", "tipo_de_operacao_da_aeronave"]
categorical_cols = [
    nome for nome, tipo in df_balanced.dtypes 
    if tipo == 'string' and nome not in colunas_ignorar
]

numeric_cols = [
    nome for nome, tipo in df_balanced.dtypes 
    if tipo in ('int', 'double') and nome not in colunas_ignorar
]

print(f"Total de colunas Categóricas (Texto) processadas: {len(categorical_cols)}")
print(f"Total de colunas Numéricas processadas: {len(numeric_cols)}")

indexers = [StringIndexer(inputCol=c, outputCol=f"{c}_indexed", handleInvalid="keep") for c in categorical_cols]
encoders = [OneHotEncoder(inputCol=f"{c}_indexed", outputCol=f"{c}_vec") for c in categorical_cols]

# Vetorizando as features preparadas
# assembler_inputs = [f"{c}_vec" for c in categorical_cols]
# vector_assembler = VectorAssembler(inputCols=assembler_inputs, outputCol="features")
assembler_inputs = [f"{c}_vec" for c in categorical_cols] + numeric_cols
vector_assembler = VectorAssembler(inputCols=assembler_inputs, outputCol="features")

scaler = StandardScaler(
    inputCol="unscaled_features", 
    outputCol="features", 
    withStd=True, 
    withMean=False
)

# Criando um pipeline de pré-processamento
pre_process_pipeline = Pipeline(stages=indexers + encoders + [vector_assembler])
df_model_ready = pre_process_pipeline.fit(df_balanced).transform(df_balanced)

Distribuição após o balanceamento das classes:
+-----+-----+
|label|count|
+-----+-----+
|  0.0|  796|
|  1.0|  821|
|  2.0|  791|
+-----+-----+

Total de colunas Categóricas (Texto) processadas: 23
Total de colunas Numéricas processadas: 5


In [45]:
df_clean.printSchema()

root
 |-- classificacao_da_ocorrencia: string (nullable = true)
 |-- uf_da_ocorrencia: string (nullable = true)
 |-- total_de_recomendacoes: integer (nullable = true)
 |-- total_de_aeronaves_envolvidas: integer (nullable = true)
 |-- ocorrencia_na_saida_da_pista: string (nullable = true)
 |-- tipo_da_ocorrencia: string (nullable = true)
 |-- tipo_de_categoria_da_ocorrencia: string (nullable = true)
 |-- taxonomia_do_tipo_de_icao: string (nullable = true)
 |-- categoria_do_operador_de_aeronave: string (nullable = true)
 |-- tipo_de_aeronave: string (nullable = true)
 |-- fabricante_da_aeronave: string (nullable = true)
 |-- modelo_de_aeronave: string (nullable = true)
 |-- tipo_de_aeronave_icao: string (nullable = true)
 |-- aeronave_motor_tipo: string (nullable = true)
 |-- aeronave_motor_quantidade: string (nullable = true)
 |-- aeronave_pmd: integer (nullable = true)
 |-- categoria_pmd_aeronave: integer (nullable = true)
 |-- registro_da_categoria_da_aeronave: string (nullable = true

In [46]:
df_clean.show(5)

+---------------------------+----------------+----------------------+-----------------------------+----------------------------+--------------------+-------------------------------+-------------------------+---------------------------------+----------------+----------------------+------------------+---------------------+-------------------+-------------------------+------------+----------------------+---------------------------------+-----------------------------+-------------------------+-----------------------+----------------------------+----------------------------+-------------------------+--------------------------------+--------------------+------------------------------+--------------------+-----------------+-----+
|classificacao_da_ocorrencia|uf_da_ocorrencia|total_de_recomendacoes|total_de_aeronaves_envolvidas|ocorrencia_na_saida_da_pista|  tipo_da_ocorrencia|tipo_de_categoria_da_ocorrencia|taxonomia_do_tipo_de_icao|categoria_do_operador_de_aeronave|tipo_de_aeronave|fabricant

## Divisão de Treino e Teste

In [47]:
# Dividindo o conjunto de dados (80% treino, 20% teste)
train_data, test_data = df_model_ready.randomSplit([0.8, 0.2], seed=42)

print(f"Registros de Treino: {train_data.count()}")
print(f"Registros de Teste: {test_data.count()}")

Registros de Treino: 1982
Registros de Teste: 426


## Treinamento

In [48]:
# --- 1. Decision Tree ---
dt_classifier = DecisionTreeClassifier(labelCol="label", featuresCol="features")
dt_model = dt_classifier.fit(train_data)
dt_predictions = dt_model.transform(test_data)

# --- 2. Logistic Regression ---
# Para multiclasse no Spark, o Logistic Regression suporta multinomial.
lr_classifier = LogisticRegression(labelCol="label", featuresCol="features", maxIter=10)
lr_model = lr_classifier.fit(train_data)
lr_predictions = lr_model.transform(test_data)

# --- 3. Redes Neurais (Multilayer Perceptron) OTIMIZADA ---
input_size = len(df_model_ready.select("features").first()[0])
output_size = len(class_counts)

# Aumentamos os neurônios para lidar com o tamanho do vetor do OneHotEncoder
# Exemplo: Entrada -> 64 neurônios -> 32 neurônios -> Saída
layers = [input_size, 64, 32, output_size]

mlp_classifier = MultilayerPerceptronClassifier(
    layers=layers, 
    labelCol="label", 
    featuresCol="features", 
    maxIter=500,        # Aumentamos de 100 para 500 iterações
    blockSize=128, 
    stepSize=0.01,      # Ajuste fino na taxa de aprendizado
    seed=42
)

print("Treinando a Rede Neural (Isso pode levar alguns minutos)...")
mlp_model = mlp_classifier.fit(train_data)
mlp_predictions = mlp_model.transform(test_data)

Treinando a Rede Neural (Isso pode levar alguns minutos)...


## Resultados

In [49]:
# Instanciando o avaliador
evaluator = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="accuracy")

# Calculando a acurácia de cada modelo
dt_accuracy = evaluator.evaluate(dt_predictions)
lr_accuracy = evaluator.evaluate(lr_predictions)
mlp_accuracy = evaluator.evaluate(mlp_predictions)

print("=== Resultados da Avaliação ===")
print(f"Decision Tree Accuracy:      {dt_accuracy:.4f}")
print(f"Logistic Regression Accuracy:{lr_accuracy:.4f}")
print(f"Neural Network Accuracy:     {mlp_accuracy:.4f}")

# Detalhamento por F1-Score (Métrica interessante para classes balanceadas)
evaluator_f1 = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="f1")

print("\n=== F1-Score ===")
print(f"Decision Tree F1:            {evaluator_f1.evaluate(dt_predictions):.4f}")
print(f"Logistic Regression F1:      {evaluator_f1.evaluate(lr_predictions):.4f}")
print(f"Neural Network F1:           {evaluator_f1.evaluate(mlp_predictions):.4f}")

=== Resultados da Avaliação ===
Decision Tree Accuracy:      0.8286
Logistic Regression Accuracy:0.7347
Neural Network Accuracy:     0.7347

=== F1-Score ===
Decision Tree F1:            0.8265
Logistic Regression F1:      0.7322
Neural Network F1:           0.7120


In [50]:
metadados_label = lr_predictions.schema["label"].metadata
nomes_classes = metadados_label["ml_attr"]["vals"]

print(f"Classes identificadas pelo modelo: {nomes_classes}\n")

# 2. Convertendo as previsões numéricas de volta para texto para facilitar a leitura
coluna_real = col("label")
coluna_prevista = col("prediction")

for i, nome in enumerate(nomes_classes):
    coluna_real = when(col("label") == i, nome).otherwise(coluna_real)
    coluna_prevista = when(col("prediction") == i, nome).otherwise(coluna_prevista)

# Adicionando essas colunas de texto ao DataFrame de previsões
preds_text = lr_predictions \
    .withColumn("Real", coluna_real) \
    .withColumn("Previsto", coluna_prevista)

# ==========================================
# 3. MATRIZ DE CONFUSÃO (PySpark DataFrame API)
# ==========================================
print("=== Matriz de Confusão (Regressão Logística) ===")
# Agrupamos pelo valor Real, pivotamos pelo Previsto e contamos.
matriz_confusao = preds_text.groupBy("Real").pivot("Previsto", nomes_classes).count().fillna(0)

# Exibindo a matriz
matriz_confusao.show()

# ==========================================
# 4. RELATÓRIO DE CLASSIFICAÇÃO (Precision, Recall, F1 por Classe)
# ==========================================
print("=== Relatório de Avaliação por Classe ===")

for i, nome in enumerate(nomes_classes):
    # Precisão (Dos que o modelo disse ser esta classe, quantos realmente eram?)
    eval_precision = MulticlassClassificationEvaluator(
        labelCol="label", predictionCol="prediction", metricName="precisionByLabel", metricLabel=float(i)
    )
    
    # Recall/Revocação (De todos os casos reais desta classe, quantos o modelo encontrou?)
    eval_recall = MulticlassClassificationEvaluator(
        labelCol="label", predictionCol="prediction", metricName="recallByLabel", metricLabel=float(i)
    )
    
    # F1-Score (Média harmônica entre Precision e Recall)
    eval_f1 = MulticlassClassificationEvaluator(
        labelCol="label", predictionCol="prediction", metricName="fMeasureByLabel", metricLabel=float(i)
    )

    # Calculando os valores
    p = eval_precision.evaluate(lr_predictions)
    r = eval_recall.evaluate(lr_predictions)
    f = eval_f1.evaluate(lr_predictions)

    # Exibindo os resultados formatados
    print(f"Classe: {nome}")
    print(f"  Precision: {p:.4f}  |  Recall: {r:.4f}  |  F1-Score: {f:.4f}\n")

Classes identificadas pelo modelo: ['INCIDENTE', 'ACIDENTE', 'INCIDENTE GRAVE']

=== Matriz de Confusão (Regressão Logística) ===
+---------------+---------+--------+---------------+
|           Real|INCIDENTE|ACIDENTE|INCIDENTE GRAVE|
+---------------+---------+--------+---------------+
|      INCIDENTE|      129|       4|             16|
|INCIDENTE GRAVE|       28|      30|             88|
|       ACIDENTE|        3|      96|             32|
+---------------+---------+--------+---------------+

=== Relatório de Avaliação por Classe ===
Classe: INCIDENTE
  Precision: 0.8063  |  Recall: 0.8658  |  F1-Score: 0.8350

Classe: ACIDENTE
  Precision: 0.7385  |  Recall: 0.7328  |  F1-Score: 0.7356

Classe: INCIDENTE GRAVE
  Precision: 0.6471  |  Recall: 0.6027  |  F1-Score: 0.6241

